# A01 — Fechas, convenciones y descuento

**Objetivos:** calendario TARGET y fecha valor; convenciones de ajuste; day counts; calendarios de pago con *stub*;
conversión entre capitalizaciones; caso de los dos préstamos.

**Fuentes:** Veronesi cap. 2; Hull cap. 4; Schofield & Bowler cap. 1. Datos sintéticos.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> repository root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

from derivlab.dates import (easter_sunday, target_holidays, is_business_day, adjust, add_business_days,
                            add_months, spot_date, year_fraction, make_schedule)

## 1. Calendario TARGET y fecha valor

In [2]:
rows = [{"year": y, "holiday": d, "weekday": d.strftime("%A")} for y in (2026, 2027) for d in sorted(target_holidays(y))]
pd.DataFrame(rows)

,year,holiday,weekday
0,2026,2026-01-01,Thursday
1,2026,2026-04-03,Friday
2,2026,2026-04-06,Monday
3,2026,2026-05-01,Friday
4,2026,2026-12-25,Friday
5,2026,2026-12-26,Saturday
6,2027,2027-01-01,Friday
7,2027,2027-03-26,Friday
8,2027,2027-03-29,Monday
9,2027,2027-05-01,Saturday


In [3]:
for trade in [date(2026, 10, 26), date(2026, 12, 23), date(2027, 3, 25), date(2027, 4, 29)]:
    print(f"trade {trade} ({trade:%a}) -> spot {spot_date(trade)} ({spot_date(trade):%a})")

trade 2026-10-26 (Mon) -> spot 2026-10-28 (Wed)
trade 2026-12-23 (Wed) -> spot 2026-12-28 (Mon)
trade 2027-03-25 (Thu) -> spot 2027-03-31 (Wed)
trade 2027-04-29 (Thu) -> spot 2027-05-03 (Mon)


## 2. Convenciones de ajuste

In [4]:
tests = [date(2026, 10, 31), date(2027, 5, 1), date(2027, 1, 1)]
pd.DataFrame([{"date": d, "weekday": f"{d:%a}",
               **{c: adjust(d, c) for c in ["following", "modified_following", "preceding"]}} for d in tests])

,date,weekday,following,modified_following,preceding
0,2026-10-31,Sat,2026-11-02,2026-10-30,2026-10-30
1,2027-05-01,Sat,2027-05-03,2027-05-03,2027-04-30
2,2027-01-01,Fri,2027-01-04,2027-01-04,2026-12-31


## 3. Day counts

In [5]:
periods = [(date(2026, 10, 28), date(2027, 4, 28)), (date(2027, 1, 31), date(2027, 2, 28)),
           (date(2027, 2, 28), date(2027, 3, 31)), (date(2027, 12, 15), date(2028, 6, 15))]
conventions = ["ACT/360", "ACT/365F", "30/360", "30E/360", "ACT/ACT ISDA"]
dc = pd.DataFrame([{"start": a, "end": b, "days": (b - a).days, **{c: year_fraction(a, b, c) for c in conventions}}
                   for a, b in periods])
dc.round(6)

,start,end,days,ACT/360,ACT/365F,30/360,30E/360,ACT/ACT ISDA
0,2026-10-28,2027-04-28,182,0.505556,0.498630,0.500000,0.500000,0.498630
1,2027-01-31,2027-02-28,28,0.077778,0.076712,0.077778,0.077778,0.076712
2,2027-02-28,2027-03-31,31,0.086111,0.084932,0.091667,0.088889,0.084932
3,2027-12-15,2028-06-15,183,0.508333,0.501370,0.500000,0.500000,0.500127


## 4. Calendarios de pago

In [6]:
trade = date(2026, 10, 26)
start = spot_date(trade)
quarterly = pd.DataFrame(make_schedule(start, add_months(start, 60), 3))
print("Quarterly 5y schedule from spot:", len(quarterly), "periods")
quarterly.head(6)

Quarterly 5y schedule from spot: 20 periods


,accrual_start,accrual_end,pay_date,fixing_date
0,2026-10-28,2027-01-28,2027-01-28,2026-10-26
1,2027-01-28,2027-04-28,2027-04-28,2027-01-26
2,2027-04-28,2027-07-28,2027-07-28,2027-04-26
3,2027-07-28,2027-10-28,2027-10-28,2027-07-26
4,2027-10-28,2028-01-28,2028-01-28,2027-10-26
5,2028-01-28,2028-04-28,2028-04-28,2028-01-26


In [7]:
stub = pd.DataFrame(make_schedule(date(2026, 10, 28), date(2029, 12, 15), 6))
stub["days"] = [(b - a).days for a, b in zip(stub.accrual_start, stub.accrual_end)]
print("Short front stub: first period has", stub.days.iloc[0], "days")
stub

Short front stub: first period has 48 days


,accrual_start,accrual_end,pay_date,fixing_date,days
0,2026-10-28,2026-12-15,2026-12-15,2026-10-26,48
1,2026-12-15,2027-06-15,2027-06-15,2026-12-11,182
2,2027-06-15,2027-12-15,2027-12-15,2027-06-11,183
3,2027-12-15,2028-06-15,2028-06-15,2027-12-13,183
4,2028-06-15,2028-12-15,2028-12-15,2028-06-13,183
5,2028-12-15,2029-06-15,2029-06-15,2028-12-13,182
6,2029-06-15,2029-12-17,2029-12-17,2029-06-13,185


## 5. Capitalizaciones

Un mismo factor de descuento admite muchas formas de escribirse:
$P = e^{-z_c t} = (1+z_m/m)^{-mt} = 1/(1 + L\tau)$.

In [8]:
def to_continuous(rate, freq):
    """freq = periods per year, or 'cont'."""
    return rate if freq == "cont" else freq * np.log(1 + rate / freq)

def from_continuous(zc, freq):
    return zc if freq == "cont" else freq * (np.exp(zc / freq) - 1)

zc = to_continuous(0.03, 1)
print({f: round(from_continuous(zc, f), 6) for f in [1, 2, 4, 12, "cont"]})
print("3% ACT/360 expressed on ACT/365F basis:", round(0.03 * 365 / 360, 6))

{1: np.float64(0.03), 2: np.float64(0.029778), 4: np.float64(0.029668), 12: np.float64(0.029595), 'cont': np.float64(0.029559)}
3% ACT/360 expressed on ACT/365F basis: 0.030417


## 6. Valor presente de un calendario de flujos

In [9]:
flows = [(d["pay_date"], 1_000_000.0) for d in make_schedule(start, add_months(start, 36), 12)]
z = 0.02
pv = sum(cf * np.exp(-z * year_fraction(trade, d, "ACT/365F")) for d, cf in flows)
print(f"PV of 3 annual flows of 1M at 2% cont: {pv:,.2f}")

PV of 3 annual flows of 1M at 2% cont: 2,882,175.65


## 7. Caso: dos ofertas de préstamo de 50 M€ a 5 años

- **Oferta A:** Euribor 3M + 1,20 %, ACT/360, pagos trimestrales.
- **Oferta B:** fijo 3,40 %, 30/360, pagos anuales.

Para compararlas hace falta un supuesto sobre el Euribor: aquí, plano al 2,05 % (en A3 se usará la curva).
Se calcula el **coste efectivo anual** de cada oferta: la tasa *y* (capitalización anual, ACT/365F)
que iguala el valor presente de los pagos al nominal recibido.

In [10]:
N, euribor3m, margin, fixed = 50_000_000, 0.0205, 0.0120, 0.0340
end_unadj = add_months(start, 60)

def floating_flows(all_in):
    per = make_schedule(start, end_unadj, 3)
    out = [(p["pay_date"], N * all_in * year_fraction(p["accrual_start"], p["accrual_end"], "ACT/360")) for p in per]
    out[-1] = (out[-1][0], out[-1][1] + N)
    return out

def fixed_flows(rate):
    per = make_schedule(start, end_unadj, 12)
    out = [(p["pay_date"], N * rate * year_fraction(p["accrual_start"], p["accrual_end"], "30/360")) for p in per]
    out[-1] = (out[-1][0], out[-1][1] + N)
    return out

def effective_rate(flows):
    f = lambda y: sum(cf * (1 + y) ** (-year_fraction(start, d, "ACT/365F")) for d, cf in flows) - N
    return brentq(f, -0.5, 0.5)

eff_a = effective_rate(floating_flows(euribor3m + margin))
eff_b = effective_rate(fixed_flows(fixed))
breakeven = brentq(lambda e: effective_rate(floating_flows(e + margin)) - eff_b, 0.0, 0.10)
int_a = sum(cf for _, cf in floating_flows(euribor3m + margin)) - N
int_b = sum(cf for _, cf in fixed_flows(fixed)) - N

print(f"CASE effective A (floating) = {eff_a:.4%}")
print(f"CASE effective B (fixed)    = {eff_b:.4%}")
print(f"CASE difference B - A       = {(eff_b - eff_a) * 1e4:.1f} bp")
print(f"CASE total interest A = {int_a:,.0f}  B = {int_b:,.0f}")
print(f"CASE break-even flat Euribor 3M = {breakeven:.4%}")

CASE effective A (floating) = 3.3361%
CASE effective B (fixed)    = 3.3980%
CASE difference B - A       = 6.2 bp
CASE total interest A = 8,242,361  B = 8,500,000
CASE break-even flat Euribor 3M = 2.1096%


**Lectura.** Con Euribor estable, la oferta A es más barata, pero el diferencial es una prima por asumir riesgo:
si el Euribor medio supera el nivel de equilibrio, la oferta B habría salido mejor. El tamaño de ese riesgo
es el tema de B1.

## 8. Comprobaciones

In [11]:
assert easter_sunday(2026) == date(2026, 4, 5)
assert spot_date(date(2026, 12, 23)) == date(2026, 12, 28)
assert abs(from_continuous(to_continuous(0.03, 4), 4) - 0.03) < 1e-12
assert abs(eff_b - 0.034) < 5e-5          # 30/360 annual with real dates is almost exactly 3.40%
assert eff_a < eff_b and euribor3m < breakeven
print("All checks passed")

All checks passed


## Ejercicios

1. **Básico.** Fracción de año entre el 31-01-2027 y el 31-07-2027 en ACT/360, 30/360 y 30E/360, e intereses de 10 M€ al 2,5 % en cada caso.
2. **Intermedio.** Genera un calendario semestral del 17-11-2026 al 15-08-2033. ¿Cuántos días tiene el *stub* y cuántos periodos hay?
3. **Avanzado.** ¿Qué margen sobre Euribor 3M (plano al 2,05 %) iguala el coste efectivo de la oferta A al de la B?

## Soluciones

In [12]:
a, b = date(2027, 1, 31), date(2027, 7, 31)
for c in ["ACT/360", "30/360", "30E/360"]:
    yf = year_fraction(a, b, c)
    print(f"{c:8s} yf = {yf:.6f}  interest = {10_000_000 * 0.025 * yf:,.2f}")

ACT/360  yf = 0.502778  interest = 125,694.44
30/360   yf = 0.500000  interest = 125,000.00
30E/360  yf = 0.500000  interest = 125,000.00


In [13]:
s = make_schedule(date(2026, 11, 17), date(2033, 8, 15), 6)
print("periods:", len(s), " stub days:", (s[0]["accrual_end"] - s[0]["accrual_start"]).days)

periods: 14  stub days: 90


In [14]:
m_star = brentq(lambda mg: effective_rate(floating_flows(euribor3m + mg)) - eff_b, 0.0, 0.05)
print(f"Break-even margin: {m_star:.4%}")

Break-even margin: 1.2596%
